In [1]:
import os
import csv
import glob
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# ==============================================================================
# 1. SPARK CON MEMORIA SUFICIENTE PARA 302 COLUMNAS
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-TurbineData-EDA") \
    .master("local[4]") \
    .config("spark.driver.memory", "12g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# ==============================================================================
# 2. EXTRAER HEADER REAL (última línea comentada con '#')
# ==============================================================================
base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")

reference_file = glob.glob(
    os.path.join(bronze_dir, "Kelmarsh_SCADA_2021_*", "Turbine_Data_Kelmarsh_1_*.csv")
)[0]

print(f"📄 Archivo de referencia: {os.path.basename(reference_file)}")

header_line = None
with open(reference_file, "r", encoding="utf-8") as f:
    for line in f:
        stripped = line.strip()
        if stripped.startswith("#"):
            header_line = stripped
        else:
            break

header_clean = header_line.lstrip("#").strip()
column_names = [c.strip() for c in next(csv.reader([header_clean]))]

print(f"✅ Columnas en header: {len(column_names)}")
print(f"   [0]  → '{column_names[0]}'")
print(f"   [1]  → '{column_names[1]}'")
print(f"   [-1] → '{column_names[-1]}'")

# ==============================================================================
# 3. CARGA DE DATOS (TURBINA 1)
# ==============================================================================
target_years = ["2021", "2022"]
selected_paths = [
    os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", "Turbine_Data_Kelmarsh_1_*.csv")
    for year in target_years
]

turbine_raw = spark.read \
    .option("header", "False") \
    .option("inferSchema", "True") \
    .option("comment", "#") \
    .csv(selected_paths)

# ==============================================================================
# 4. RENOMBRAR COLUMNAS CON NOMBRES REALES
# ==============================================================================
n_csv = len(turbine_raw.columns)
n_hdr = len(column_names)

if n_csv == n_hdr:
    turbine_df = turbine_raw.toDF(*column_names)
    print(f"\n✅ Renombrado correcto: {n_csv} columnas")
else:
    print(f"\n⚠️  Mismatch: CSV={n_csv} cols vs Header={n_hdr} cols — usando nombres raw")
    turbine_df = turbine_raw

# ==============================================================================
# 5. CACHE Y VOLUMETRÍA
# ==============================================================================
turbine_df.cache()
total_records = turbine_df.count()   # Materializa el cache
total_cols    = len(turbine_df.columns)

print(f"\n🎯 Registros totales (Turbina 1): {total_records:,}")
print(f"📐 Número de columnas: {total_cols}")

# ==============================================================================
# 6. SCHEMA CON NOMBRES REALES
# ==============================================================================
print("\n📋 SCHEMA CON NOMBRES REALES:")
turbine_df.printSchema()

# ==============================================================================
# 7. PREVIEW
# ==============================================================================
print("\n👁️  PRIMERAS 3 FILAS:")
turbine_df.show(3, truncate=40)

# ==============================================================================
# 8. ESTADÍSTICAS DESCRIPTIVAS EN LOTES (evita OOM con 302 columnas)
# ==============================================================================
numeric_cols = [
    f.name for f in turbine_df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

BATCH_SIZE  = 20
total_batches = -(-len(numeric_cols) // BATCH_SIZE)   # ceil division
all_stats   = []

print(f"\n📊 ESTADÍSTICAS DESCRIPTIVAS — {len(numeric_cols)} columnas numéricas "
      f"(procesando en {total_batches} lotes de {BATCH_SIZE}):")

for i in range(0, len(numeric_cols), BATCH_SIZE):
    batch = numeric_cols[i : i + BATCH_SIZE]
    print(f"  📦 Lote {i//BATCH_SIZE + 1}/{total_batches}: cols {i}–{i+len(batch)-1}")

    alias_map = {name: f"c{j}" for j, name in enumerate(batch)}

    batch_df = turbine_df.select([
        F.col(f"`{name}`").alias(alias_map[name]) for name in batch
    ]).summary("count", "mean", "stddev", "min", "25%", "75%", "max")

    batch_pd = batch_df.toPandas()
    batch_pd.columns = ["summary"] + batch
    all_stats.append(batch_pd.set_index("summary").T)

stats_final = pd.concat(all_stats)
stats_final.index.name = "column"

print("\n" + stats_final.to_string())

# ==============================================================================
# 9. NULOS / NaN POR COLUMNA EN LOTES
# ==============================================================================
all_nulls = []
total_batches_all = -(-len(turbine_df.columns) // BATCH_SIZE)

print(f"\n🚨 NULOS / NaN — procesando {len(turbine_df.columns)} columnas "
      f"en {total_batches_all} lotes:")

for i in range(0, len(turbine_df.columns), BATCH_SIZE):
    batch_fields = turbine_df.schema.fields[i : i + BATCH_SIZE]
    print(f"  📦 Lote {i//BATCH_SIZE + 1}/{total_batches_all}: cols {i}–{i+len(batch_fields)-1}")

    null_exprs = []
    for f in batch_fields:
        escaped = f"`{f.name}`"
        if isinstance(f.dataType, (DoubleType, FloatType)):
            expr = F.count(F.when(
                F.col(escaped).isNull() | F.isnan(F.col(escaped)), 1
            )).alias(f.name)
        else:
            expr = F.count(F.when(
                F.col(escaped).isNull(), 1
            )).alias(f.name)
        null_exprs.append(expr)

    batch_nulls = turbine_df.select(null_exprs).toPandas().T.reset_index()
    batch_nulls.columns = ["column", "null_or_nan"]
    all_nulls.append(batch_nulls)

nulls_final = pd.concat(all_nulls, ignore_index=True)
nulls_final["pct"] = (nulls_final["null_or_nan"] / total_records * 100).round(2)
nulls_final = nulls_final.sort_values("pct", ascending=False)

with_nulls  = nulls_final[nulls_final["null_or_nan"] > 0]
clean_cols  = (nulls_final["null_or_nan"] == 0).sum()

print(f"\nColumnas con nulos/NaN : {len(with_nulls)} de {len(nulls_final)}")
print(f"Columnas completamente limpias: {clean_cols}\n")
print(with_nulls.to_string(index=False))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/30 11:18:39 WARN Utils: Your hostname, medion, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlo1)
26/05/30 11:18:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 11:18:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📄 Archivo de referencia: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
✅ Columnas en header: 303
   [0]  → 'Date and time'
   [1]  → 'Wind speed (m/s)'
   [-1] → 'Tower Acceleration Y, StdDev (mm/ss)'



✅ Renombrado correcto: 303 columnas


26/05/30 11:18:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



🎯 Registros totales (Turbina 1 · 2021-2024): 105,120
📐 Número de columnas: 303

📋 SCHEMA CON NOMBRES REALES:
root
 |-- Date and time: timestamp (nullable = true)
 |-- Wind speed (m/s): double (nullable = true)
 |-- Wind speed, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed, Minimum (m/s): double (nullable = true)
 |-- Wind speed, Maximum (m/s): double (nullable = true)
 |-- Long Term Wind (m/s): double (nullable = true)
 |-- Wind speed Sensor 1 (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Minimum (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Maximum (m/s): double (nullable = true)
 |-- Wind speed Sensor 2 (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Minimum (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Maximum (m/s): double (nullable = true)
 |-- Density adjusted w

...

  📦 Lote 6/16: cols 100–119
  📦 Lote 7/16: cols 120–139
  📦 Lote 8/16: cols 140–159
  📦 Lote 9/16: cols 160–179
  📦 Lote 10/16: cols 180–199
  📦 Lote 11/16: cols 200–219
  📦 Lote 12/16: cols 220–239


  📦 Lote 13/16: cols 240–259
  📦 Lote 14/16: cols 260–279
  📦 Lote 15/16: cols 280–299


  📦 Lote 16/16: cols 300–301

summary                                                    count                   mean                 stddev                     min                     25%                    75%                 max
column                                                                                                                                                                                                   
Wind speed (m/s)                                          105120                    NaN                    NaN                     0.0      3.8344810247421273       7.56212797164917                 NaN
Wind speed, Standard deviation (m/s)                      105120                    NaN                    NaN                     0.0       0.558426271576452     1.1257791655373184                 NaN
Wind speed, Minimum (m/s)                                 105120                    NaN                    NaN       -14.9624996185303        2.45402479171753    

Incorrect Java version: 21.0.10+7-Ubuntu-124.04
jspawnhelper version 21.0.11+10-1-24.04.2-Ubuntu
This command is not for general use and should only be run as the result of a call to
ProcessBuilder.start() or Runtime.exec() in a java application
Incorrect Java version: 21.0.10+7-Ubuntu-124.04
jspawnhelper version 21.0.11+10-1-24.04.2-Ubuntu
This command is not for general use and should only be run as the result of a call to
ProcessBuilder.start() or Runtime.exec() in a java application
Incorrect Java version: 21.0.10+7-Ubuntu-124.04
jspawnhelper version 21.0.11+10-1-24.04.2-Ubuntu
This command is not for general use and should only be run as the result of a call to
ProcessBuilder.start() or Runtime.exec() in a java application
Incorrect Java version: 21.0.10+7-Ubuntu-124.04
jspawnhelper version 21.0.11+10-1-24.04.2-Ubuntu
This command is not for general use and should only be run as the result of a call to
ProcessBuilder.start() or Runtime.exec() in a java application
Incorrect Java v

26/05/30 12:17:46 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-e64959aa-7aa7-4a2b-895d-08d51cd5e01a. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-e64959aa-7aa7-4a2b-895d-08d51cd5e01a
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:344)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:372)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:368)
	at scala.collection.ArrayOps$.foreach